In [1]:
!pip install -q openai-whisper
!pip install -q librosa
!pip install -q soundfile
!pip install -q gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.6 MB/s eta 0:00:00


In [2]:
import whisper
import re
import gradio as gr
from collections import Counter

In [3]:
print("Loading Whisper model...")

whisper_model = whisper.load_model("base")

print("Whisper model loaded successfully!")

Loading Whisper model...


100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 84.2MiB/s]


Whisper model loaded successfully!


In [4]:
def speech_to_text(audio_file):

    result = whisper_model.transcribe(audio_file)

    transcript = result["text"].strip()

    return transcript

In [5]:
news_keywords = [
    "government",
    "minister",
    "president",
    "election",
    "economy",
    "business",
    "technology",
    "science",
    "education",
    "health",
    "sports",
    "crime",
    "police",
    "accident",
    "weather",
    "environment",
    "company",
    "market",
    "international",
    "India"
]


def analyze_keywords(text):

    words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())

    word_count = Counter(words)

    found_keywords = {}

    for keyword in news_keywords:

        count = word_count[keyword.lower()]

        if count > 0:
            found_keywords[keyword] = count

    return found_keywords

In [6]:
positive_words = [
    "success",
    "successful",
    "growth",
    "improve",
    "improved",
    "benefit",
    "good",
    "positive",
    "win",
    "won",
    "achievement",
    "progress",
    "increase",
    "peace"
]

negative_words = [
    "loss",
    "lost",
    "crisis",
    "accident",
    "death",
    "dead",
    "crime",
    "violence",
    "attack",
    "problem",
    "negative",
    "decline",
    "decrease",
    "danger"
]


def analyze_sentiment(text):

    words = re.findall(r'\b\w+\b', text.lower())

    positive_count = sum(
        1 for word in words
        if word in positive_words
    )

    negative_count = sum(
        1 for word in words
        if word in negative_words
    )

    if positive_count > negative_count:
        sentiment = "Positive 😊"

    elif negative_count > positive_count:
        sentiment = "Negative 😟"

    else:
        sentiment = "Neutral 😐"

    return {
        "sentiment": sentiment,
        "positive_count": positive_count,
        "negative_count": negative_count
    }

In [7]:
def analyze_news_text(text):

    words = re.findall(r'\b\w+\b', text)

    sentences = re.split(r'[.!?]+', text)

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    return {
        "word_count": len(words),
        "sentence_count": len(sentences)
    }

In [8]:
def generate_summary(text):

    sentences = re.split(r'[.!?]+', text)

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    if len(sentences) == 0:
        return "No summary available."

    # Take the first 3 sentences as a simple summary
    summary_sentences = sentences[:3]

    summary = ". ".join(summary_sentences)

    return summary + "."

In [9]:
def news_analyzer(audio_file, text_input):

    # Get news text
    if audio_file is not None:
        transcript = speech_to_text(audio_file)

    elif text_input and text_input.strip():
        transcript = text_input.strip()

    else:
        return (
            "Please enter news text or upload/record news audio.",
            "",
            "",
            "",
            ""
        )

    if not transcript:
        return (
            "Could not understand the news.",
            "",
            "",
            "",
            ""
        )

    # Analyze text
    text_result = analyze_news_text(transcript)

    # Analyze keywords
    keywords = analyze_keywords(transcript)

    # Analyze sentiment
    sentiment_result = analyze_sentiment(transcript)

    # Generate summary
    summary = generate_summary(transcript)

    # Keyword report
    if keywords:
        keyword_report = "\n".join(
            f"• {key}: {value}"
            for key, value in keywords.items()
        )
    else:
        keyword_report = "No important news keywords detected."

    # Analysis report
    report = (
        f"Word Count: {text_result['word_count']}\n"
        f"Sentence Count: {text_result['sentence_count']}\n"
        f"Positive Words: {sentiment_result['positive_count']}\n"
        f"Negative Words: {sentiment_result['negative_count']}"
    )

    return (
        transcript,
        summary,
        keyword_report,
        sentiment_result["sentiment"],
        report
    )

In [10]:
with gr.Blocks(title="News Text & Speech Analyzer") as app:

    gr.Markdown("""
    # 📰 News Text & Speech Analyzer

    ### Analyze news using Text and Speech Analysis

    **Text / Speech → ASR → Keywords → Sentiment → Summary → Report**
    """)

    with gr.Row():

        with gr.Column():

            text_input = gr.Textbox(
                label="📝 Enter News Text",
                placeholder="Paste or type a news article here...",
                lines=8
            )

            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="🎤 Or Record / Upload News Audio"
            )

            analyze_button = gr.Button(
                "🔍 Analyze News",
                variant="primary"
            )

        with gr.Column():

            transcript_output = gr.Textbox(
                label="📝 News Text",
                lines=8
            )

            summary_output = gr.Textbox(
                label="📌 News Summary",
                lines=5
            )

            keywords_output = gr.Textbox(
                label="🔑 Important Keywords",
                lines=6
            )

            sentiment_output = gr.Textbox(
                label="😊 News Sentiment"
            )

            report_output = gr.Textbox(
                label="📊 News Analysis Report",
                lines=6
            )


    analyze_button.click(
        fn=news_analyzer,
        inputs=[audio_input, text_input],
        outputs=[
            transcript_output,
            summary_output,
            keywords_output,
            sentiment_output,
            report_output
        ]
    )


app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59d7055ecc1152e4c2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
